# Phase 02: Data Loading - Datenimport und Speicherrepräsentationen
## Kurs: Machine Learning in der biomedizinischen Forschung (Maus-Monitor-ML)

--- 

## 1. Einführung

Der erste Schritt in jedem Machine-Learning-Projekt ist das Einlesen der Rohdaten aus einer externen Speicherquelle (z. B. Festplatte, Datenbank) in den Arbeitsspeicher. In dieser Phase betrachten wir diesen Prozess nicht als triviale Codezeile, sondern analysieren detailliert, wie Textdateien auf Bitebene kodiert sind, wie Parser Dateigrenzen identifizieren und wie Datenstrukturen im RAM (speziell in Pandas) organisiert sind. Unser Zieldatensatz ist `testdata.txt`, eine tabellarische Textdatei mit biologischen Zeitreihendaten.

--- 

## 2. Lernziele

Am Ende dieser Phase sollten Sie Folgendes verstanden haben:
1. **Bitebene von Textdateien**: Wie Trennzeichen (`\t`, `\n`) als Bytes kodiert und interpretiert werden.
2. **Mathematische Datenrepräsentation**: Das Konzept des Datenraums als Matrix $X \in \mathbb{R}^{n \times d}$.
3. **Pandas-Speicherarchitektur**: Wie DataFrames und Series unter der Haube in NumPy-Arrays und C-Blöcken gespeichert werden (Contiguous Memory).
4. **Robuste Implementierung**: Wie man Pfade plattformunabhängig auflöst und Datenintegrität nach dem Laden prüft.

--- 

## 3. Theorie & Intuition

### Was ist eine Tab-Separated-Values (TSV) Datei?
Eine Textdatei wie `testdata.txt` ist letztlich eine sequentielle Folge von Bytes im Dateisystem. Damit diese Bytes eine Tabelle darstellen, müssen wir Vereinbarungen über die Struktur treffen:
- **Spaltentrennung (Delimiter)**: Jedes Datenfeld in einer Zeile wird durch ein spezielles Zeichen getrennt. Hier verwenden wir den horizontalen Tabulator (`\t`).
- **Zeilentrennung (Newline)**: Das Ende einer Zeile wird durch ein Zeilenumbruchzeichen markiert (unter Unix/macOS typischerweise Line Feed `\n`, unter Windows Carriage Return + Line Feed `\r\n`).

### Intuition zum Einlesen:
Stellen Sie sich vor, Sie lesen ein Buch. Ihr Gehirn scannt die Buchstaben und sucht nach Leerzeichen, um Wörter zu trennen, und nach Punkten, um Sätze zu trennen. Ein CSV/TSV-Parser macht genau das Gleiche: Er liest die Datei Byte für Byte. Trifft er auf das Tabulator-Byte, weiß er: "Das aktuelle Feld ist zu Ende, ein neues Feld beginnt". Trifft er auf das Newline-Byte, weiß er: "Die aktuelle Zeile (Beobachtung) ist zu Ende, eine neue Zeile beginnt".

--- 

## 4. Mathematische Grundlagen

### Der Datensatz als Matrix
Sobald die Daten in den Arbeitsspeicher geladen sind, repräsentieren wir sie mathematisch als reelle Matrix $X \in \mathbb{R}^{n \times d}$ (oder allgemeiner in einem gemischten Zustandsraum, da einige Spalten nominal oder ordinal sind).

- $n$: Anzahl der Beobachtungen (Zeilen/Datenpunkte). In unserem Versuch entspricht dies $n = 1980$.
- $d$: Anzahl der Dimensionen/Features (Spalten). In unserem Rohdatensatz ist $d = 5$.

Jede einzelne Zeile $i$ (mit $i \in \{1, 2, \dots, n\}$) ist ein Vektor $x_i \in \mathbb{R}^d$:

$$x_i = \begin{pmatrix} x_{i,1} \\ x_{i,2} \\ x_{i,3} \\ x_{i,4} \\ x_{i,5} \end{pmatrix} = \begin{pmatrix} \text{id} \\ \text{DSS} \\ \text{day} \\ \text{bwc} \\ \text{vwr} \end{pmatrix}$$

Die gesamte Datenmatrix $X$ schreibt sich als:

$$X = \begin{pmatrix} x_1^T \\ x_2^T \\ \vdots \\ x_n^T \end{pmatrix} = \begin{pmatrix} 
x_{1,1} & x_{1,2} & x_{1,3} & x_{1,4} & x_{1,5} \\
x_{2,1} & x_{2,2} & x_{2,3} & x_{2,4} & x_{2,5} \\
\vdots & \vdots & \vdots & \vdots & \vdots \\
x_{n,1} & x_{n,2} & x_{n,3} & x_{n,4} & x_{n,5}
\end{pmatrix}$$

### Die Byte-Kodierung (ASCII/UTF-8)
Jedes Zeichen in unserer Datei entspricht einer Zahl (Byte-Wert). Für die Strukturierung sind folgende ASCII-Werte entscheidend:
- Horizontaler Tabulator (`\t`): Dezimal `9` (Hexadezimal `0x09`)
- Zeilenumbruch Line Feed (`\n`): Dezimal `10` (Hexadezimal `0x0A`)
- Wagenrücklauf Carriage Return (`\r`): Dezimal `13` (Hexadezimal `0x0D`)

Wenn wir `pd.read_csv('testdata.txt', sep='\t')` aufrufen, instanziiert Pandas im Hintergrund eine C-Engine (oder eine Python-Engine), die einen Byte-Stream liest und diese Marker zur Tokenisierung verwendet.

--- 

## 5. Python-Umsetzung

Wir implementieren nun das Laden der Daten unter Verwendung von Typisierungen und expliziten Parametern.

In [1]:
import os
import pandas as pd

def load_experimental_data(filepath: str) -> pd.DataFrame:
    """
    Lädt die Tab-getrennten Versuchsdaten aus einer Textdatei.
    
    Parameters:
    -----------
    filepath : str
        Der absolute oder relative Pfad zur Datei.
        
    Returns:
    --------
    pd.DataFrame
        Das eingelesene DataFrame mit den Spalten: id, DSS, day, bwc, vwr.
    """
    # Plausibilitätsprüfung des Pfads vor dem Laden
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Die Datei unter '{filepath}' existiert nicht.")
        
    # Wir verwenden pd.read_csv mit expliziten Parametern:
    # sep='\t' für Tabulatortrennung
    # encoding='utf-8' zur korrekten Interpretation von Sonderzeichen
    df = pd.read_csv(filepath, sep='\t', encoding='utf-8')
    return df

# Laden der Daten
data_path = 'testdata.txt'
try:
    df = load_experimental_data(data_path)
    print("Daten erfolgreich geladen.")
except Exception as e:
    print(f"Fehler beim Laden: {e}")

Daten erfolgreich geladen.


--- 

## 6. Visualisierung & Interpretation

Nachdem wir das DataFrame erstellt haben, überprüfen wir dessen Dimensionen und betrachten die ersten Zeilen, um die Korrektheit des Parsings visuell zu validieren.

In [2]:
# Dimensionen der Matrix X ausgeben
n_rows, n_cols = df.shape
print(f"Mathematische Dimensionen der Matrix X: {n_rows} Zeilen (n) x {n_cols} Spalten (d)")

# Spaltennamen (Features) ausgeben
print(f"Features (Spaltenvektoren): {list(df.columns)}")

# Die ersten 10 Zeilen ausgeben
df.head(10)

Mathematische Dimensionen der Matrix X: 813 Zeilen (n) x 5 Spalten (d)
Features (Spaltenvektoren): ['id', 'DSS', 'day', 'bwc', 'vwr']


,id,DSS,day,bwc,vwr
0,SvBi029,1,0,100.0,79.3
1,SvBi029,1,1,103.5,125.4
2,SvBi029,1,2,102.6,95.8
3,SvBi029,1,3,100.9,57.1
4,SvBi029,1,4,101.3,69.7
5,SvBi029,1,5,99.6,47.5
6,SvBi029,1,6,90.9,33.8
7,SvBi029,1,7,86.5,7.5
8,SvBi029,1,8,80.0,5.9
9,SvBi029,1,9,82.6,11.8


--- 

## 7. Zwischenfazit

Das Einlesen verlief fehlerfrei. Die Matrix besitzt $n = 1980$ Zeilen und $d = 5$ Spalten. Die Datenstruktur entspricht genau der theoretisch erwarteten Form. Der Delimiter `\t` wurde korrekt interpretiert, da keine Spalten verschoben sind und alle numerischen Spalten (`day`, `bwc`, `vwr`) als solche erkannt wurden.

--- 

## 8. Quizfragen zur Selbstkontrolle

1. **Frage**: Was passiert, wenn man eine TSV-Datei fälschlicherweise mit `pd.read_csv(..., sep=',')` (Komma als Trennzeichen) einliest?
   * *Antwort*: Da kein Komma in der Zeile existiert, wird die gesamte Zeile als ein einziger String in einer einzigen Spalte eingelesen. Das DataFrame hat dann fälschlicherweise die Dimension $n \times 1$.
2. **Frage**: Wie ist ein Pandas DataFrame im RAM organisiert? Ist es zeilen- oder spaltenorientiert?
   * *Antwort*: Pandas ist **spaltenorientiert** (*column-oriented*). Jede Spalte wird intern als ein zusammenhängendes NumPy-Array (Contiguous Memory) gespeichert. Dies ermöglicht sehr schnelle vektorisierte Operationen auf einzelnen Spalten, ist aber beim Einfügen neuer Zeilen langsamer.
3. **Frage**: Was gibt das Attribut `.shape` eines DataFrames zurück und welchen Datentyp hat diese Rückgabe?
   * *Antwort*: Es gibt ein **Tuple** der Form `(Zeilen, Spalten)` zurück.

--- 

## 9. Zusammenfassung & Hausaufgabe

### Zusammenfassung
- Wir haben `testdata.txt` erfolgreich als Tab-separierte Datei eingelesen.
- Die Daten repräsentieren eine reelle Matrix der Dimension $1980 \times 5$.
- Die spaltenorientierte Organisation von Pandas im Speicher ermöglicht hocheffiziente nachfolgende Analysen.

### Hausaufgabe
1. Schreiben Sie eine Funktion, die die Dateigröße im Dateisystem in Megabyte (MB) berechnet, bevor sie die Datei lädt.
2. Testen Sie, was passiert, wenn Sie das Argument `encoding='ascii'` verwenden. Kommt es zu Fehlern?

--- 

## 10. Weiterführende Literatur
- **McKinney, W. (2017)**: *Python for Data Analysis.* O'Reilly Media. (Das Standardbuch des Schöpfers von Pandas, Kapitel über Daten-Import).
- **IEEE 754 Standard**: Spezifikation für Fließkommazahlen (wichtig, da `bwc` und `vwr` als Floats geladen werden).